# exact_DW_GS_benchmark

Exact equilibrium domain-wall GS benchmarking notebook with CPU/thread caps, half-system contour diagnostics, integrated contour scans, and cached dynamic-state comparisons.

## Setup

Minimal path configuration so the notebook can import project code from `src/`.


In [ ]:
# Minimal path setup for imports from `src`
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


## Runtime Controls

Set the CPU cap and BLAS thread limits before importing the heavier numerical stack.


In [ ]:
# CPU / BLAS thread controls
# Run this before importing numpy/scipy/matplotlib.
import os
import multiprocessing as mp

CPU_CAP = 20
AFFINITY_START = 90  # e.g. 60 to pin to a contiguous block, or None to skip

cpu_available = mp.cpu_count()
cpu_use = max(1, min(int(CPU_CAP), cpu_available))

os.environ["MY_CPU_COUNT"] = str(cpu_use)
for _k in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "NUMEXPR_MAX_THREADS",
):
    os.environ[_k] = str(cpu_use)

if AFFINITY_START is not None:
    try:
        os.sched_setaffinity(0, set(range(int(AFFINITY_START), int(AFFINITY_START) + cpu_use)))
        print(f"CPU affinity set to cores [{AFFINITY_START}, {AFFINITY_START + cpu_use - 1}]")
    except Exception as exc:
        print(f"CPU affinity not set: {exc}")

print(f"CPU_CAP={CPU_CAP}, cpu_available={cpu_available}, cpu_use={cpu_use}")

## Shared Imports and Helpers

Load plotting/numerics plus reusable helper functions used by all downstream benchmark cells.


In [ ]:
import json
from pathlib import Path
import importlib

import matplotlib.pyplot as plt
import numpy as np

classA_U1FGTN_module = importlib.import_module("fgtn.classA_U1FGTN")
importlib.reload(classA_U1FGTN_module)
classA_U1FGTN = classA_U1FGTN_module.classA_U1FGTN


np.set_printoptions(precision=5, suppress=True)
plt.rcParams["figure.dpi"] = 120


def contiguous_y_block(ny, length, start=0):
    start = int(start) % ny
    length = int(length)
    if length < 1 or length > ny:
        raise ValueError(f"length must lie in [1, {ny}], got {length}")
    return (np.arange(length, dtype=int) + start) % ny


def contiguous_x_block(nx, start, stop_inclusive):
    start = int(start) % nx
    stop_inclusive = int(stop_inclusive) % nx
    if start <= stop_inclusive:
        return np.arange(start, stop_inclusive + 1, dtype=int)
    return np.concatenate([np.arange(start, nx, dtype=int), np.arange(0, stop_inclusive + 1, dtype=int)])


def subsystem_indices(nx, ny, x_vals, y_vals):
    x_vals = np.asarray(x_vals, dtype=int)
    y_vals = np.asarray(y_vals, dtype=int)
    idx = []
    for y in y_vals:
        base_y = 2 * nx * int(y)
        for x in x_vals:
            x = int(x) % nx
            idx.append(base_y + 2 * x)
            idx.append(base_y + 2 * x + 1)
    return np.asarray(idx, dtype=int)


def reduced_covariance(G_top, idx):
    return np.asarray(G_top, dtype=np.complex128)[np.ix_(idx, idx)]


def contour_for_subsystem(model, G_top, x_vals, y_vals):
    idx = subsystem_indices(model.Nx, model.Ny, x_vals, y_vals)
    G_sub = reduced_covariance(G_top, idx)
    s_map = model.entanglement_contour(G_sub, len(x_vals), len(y_vals))
    return G_sub, s_map


def load_final_batch(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    with np.load(path) as data:
        if "G_final" in data:
            arr = np.asarray(data["G_final"], dtype=np.complex128)
        elif "G_hist" in data:
            hist = np.asarray(data["G_hist"], dtype=np.complex128)
            if hist.ndim == 4:
                arr = hist[:, -1]
            elif hist.ndim == 3:
                arr = hist
            else:
                raise ValueError(f"'G_hist' in {path} must be 3D or 4D, got {hist.shape}")
        else:
            raise KeyError(f"Expected 'G_final' or 'G_hist' in {path}, found keys={list(data.keys())}")
    if arr.ndim == 2:
        arr = arr[None, ...]
    if arr.ndim != 3 or arr.shape[1] != arr.shape[2]:
        raise ValueError(f"Expected (S, N, N) finals from {path}, got {arr.shape}")
    return arr


def frobenius_norm(M):
    M = np.asarray(M)
    return float(np.sqrt(np.sum(np.abs(M) ** 2)))


def frobenius_norm_batch(batch):
    batch = np.asarray(batch)
    return np.sqrt(np.sum(np.abs(batch) ** 2, axis=(1, 2)))


def sitewise_difference_map(delta, nx, ny):
    delta = np.asarray(delta, dtype=np.complex128)
    delta6 = delta.reshape(2, nx, ny, 2, nx, ny, order="F")
    return np.sqrt(np.sum(np.abs(delta6) ** 2, axis=(0, 3, 4, 5))).real


def unflatten_to_G6(Gfull, Nx, Ny, top=True):
    """
    Take a flattened top-layer matrix of shape (2*Nx*Ny, 2*Nx*Ny) and reshape it to
    G6 of shape (Nx, Ny, 2, Nx, Ny, 2).
    """
    if top:
        Nlayer = 2 * Nx * Ny
        if Gfull.shape != (Nlayer, Nlayer):
            raise ValueError(f"Expected ({Nlayer}, {Nlayer}), got {Gfull.shape}")

    G6m = np.asarray(Gfull, dtype=np.complex128).reshape((2, Nx, Ny, 2, Nx, Ny), order="F")
    return np.transpose(G6m, (1, 2, 0, 4, 5, 3))


def _flatten_full_matrix(G6):
    """
    Flatten G(x, y, mu; x', y', nu) using i = mu + 2*x + 2*Nx*y.
    """
    G6 = np.asarray(G6, dtype=np.complex128)
    Nx, Ny, s1, Nx2, Ny2, s2 = G6.shape
    if not ((s1, s2) == (2, 2) and (Nx, Ny) == (Nx2, Ny2)):
        raise ValueError(f"G must have shape (Nx, Ny, 2, Nx, Ny, 2); got {G6.shape}")
    G6m = np.transpose(G6, (2, 0, 1, 5, 3, 4))
    return G6m.reshape(2 * Nx * Ny, 2 * Nx * Ny, order="F")


def _fft_y_blocks_from_flat(G6, norm="ortho", hermitize=True):
    """
    Go to k_y basis via unitary FFT on y and IFFT on y', then flatten and
    extract Ny contiguous blocks of size (2*Nx) x (2*Nx).
    """
    G6 = np.asarray(G6, dtype=np.complex128)
    Nx, Ny, s1, Nx2, Ny2, s2 = G6.shape
    if not ((s1, s2) == (2, 2) and (Nx, Ny) == (Nx2, Ny2)):
        raise ValueError(f"G must have shape (Nx, Ny, 2, Nx, Ny, 2); got {G6.shape}")

    Gk6 = np.fft.fft(G6, axis=1, norm=norm)
    Gk6 = np.fft.ifft(Gk6, axis=4, norm=norm)

    G_flat = _flatten_full_matrix(Gk6)
    if hermitize:
        G_flat = 0.5 * (G_flat + G_flat.conj().T)

    blocks = []
    for j in range(Ny):
        lo = 2 * j * Nx
        hi = 2 * (j + 1) * Nx
        Bj = G_flat[lo:hi, lo:hi]
        if hermitize:
            Bj = 0.5 * (Bj + Bj.conj().T)
        blocks.append(Bj)

    ky_vals = 2 * np.pi * np.fft.fftfreq(Ny, d=1.0)
    return ky_vals, blocks


def plot_ky_spectrum(G6, norm="ortho", hermitize=True, ax=None, marker=".", ms=3):
    """
    FFT to k_y, extract contiguous 2*Nx blocks, diagonalize each block, and plot
    the ordered eigenvalues versus k_y.
    """
    ky_vals, blocks = _fft_y_blocks_from_flat(G6, norm=norm, hermitize=hermitize)

    G6 = np.asarray(G6)
    Nx, Ny = int(G6.shape[0]), int(G6.shape[1])

    evals = np.empty((Ny, 2 * Nx), dtype=float)
    for j, B in enumerate(blocks):
        w = np.linalg.eigvalsh(B)
        evals[j, :] = np.sort(np.real_if_close(w))

    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4.2))

    for band in evals.T:
        ax.plot(ky_vals, band, marker, ms=ms, lw=0)

    ax.set_xlabel(r"$k_y$")
    ax.set_ylabel("eigenvalue")
    ax.set_title(r"Spectrum of $G(k_y)$ blocks (contiguous $2N_x$ rule)")
    ax.grid(True, alpha=0.3)
    return ky_vals, evals, ax


## 1. Construct the equilibrium DW Chern-insulator GS

In [ ]:
# User configuration for the exact equilibrium DW GS
NX, NY = 20, 40
ALPHA_1, ALPHA_2 = 30, 1
PERIODIC = True
TRIV_REGION_LOCAL_MODE = False

model = classA_U1FGTN(NX, NY, nshell=None, DW=True, alpha_1=ALPHA_1, alpha_2=ALPHA_2)
H_eq = model._domain_wall_hamiltonian(
    periodic=PERIODIC,
    triv_region_local_mode=TRIV_REGION_LOCAL_MODE,
)
evals_eq = np.linalg.eigvalsh(H_eq)
G_eq = model.G_CI_domain_wall(
    periodic=PERIODIC,
    triv_region_local_mode=TRIV_REGION_LOCAL_MODE,
)

neg_count = int(np.count_nonzero(evals_eq < 0.0))
pos_count = int(np.count_nonzero(evals_eq > 0.0))
spectral_gap = float(np.min(np.abs(evals_eq)))

H_eq6 = unflatten_to_G6(H_eq, NX, NY, top=True)

# Build the k_y-resolved block spectrum and shift to a monotone k_y axis
ky_vals, ky_blocks = _fft_y_blocks_from_flat(H_eq6, norm="ortho", hermitize=True)
ky_evals = np.empty((NY, 2 * NX), dtype=float)
for j, B in enumerate(ky_blocks):
    ky_evals[j, :] = np.sort(np.real_if_close(np.linalg.eigvalsh(B)))

ky_vals = np.fft.fftshift(ky_vals)
ky_evals = np.fft.fftshift(ky_evals, axes=0)
ky_vals_pi = ky_vals / np.pi

dw_loc = getattr(model, "DW_loc", None)

print(f"System size: Nx={NX}, Ny={NY}")
print(f"alpha_1={ALPHA_1}, alpha_2={ALPHA_2}, periodic={PERIODIC}")
print(f"triv_region_local_mode={TRIV_REGION_LOCAL_MODE}")
print(f"DW locations: {dw_loc}")
print(f"H_eq shape: {H_eq.shape}, G_eq shape: {G_eq.shape}")
print(f"negative modes={neg_count}, positive modes={pos_count}, min |E|={spectral_gap:.6e}")

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), constrained_layout=True)

# 1) Full eigenspectrum vs eigenvalue index
axes[0].plot(np.arange(evals_eq.size), evals_eq, marker=".", ls="none", ms=3, color="blue")
axes[0].axhline(0.0, color="k", lw=1, alpha=0.5)
axes[0].set_title("DW Hamiltonian eigenspectrum")
axes[0].set_xlabel("eigenvalue index")
axes[0].set_ylabel("energy")
axes[0].grid(alpha=0.3)

textbox = (
    rf"$\alpha_{{\mathrm{{triv}}}} = {ALPHA_1}$, $\alpha_{{\mathrm{{top}}}} = {ALPHA_2}$" "\n"
    rf"DWs at $x = {dw_loc}$" "\n"
    rf"$N_x={NX}, N_y={NY}$"
)
axes[0].text(
    0.03,
    0.97,
    textbox,
    transform=axes[0].transAxes,
    ha="left",
    va="top",
    fontsize=11,
    linespacing=2,
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.8),
)

# 2) Energy vs k_y, with k_y shown in units of pi
for band in ky_evals.T:
    axes[1].plot(ky_vals_pi, band, ".", ms=3, color="blue")

axes[1].set_title(r"Energy spectrum vs $k_y/\pi$")
axes[1].set_xlabel(r"$k_y/\pi$")
axes[1].set_ylabel("energy")
axes[1].set_xlim(-1.0, 1.0)
axes[1].set_xticks([-1.0, -0.5, 0.0, 0.5, 1.0])
axes[1].grid(alpha=0.3)

# 3) Zoomed energy vs k_y in a tunable energy window
ENERGY_WINDOW = (-3.0, 3.0)  # Tunable energy window for the zoomed k_y spectrum

for band in ky_evals.T:
    axes[2].plot(ky_vals_pi, band, ".", ms=3, color="blue")

axes[2].set_title(rf"Energy spectrum vs $k_y/\pi$ ({ENERGY_WINDOW[0]:.0f} to {ENERGY_WINDOW[1]:.0f})")
axes[2].set_xlabel(r"$k_y/\pi$")
axes[2].set_ylabel("energy")
axes[2].set_xlim(-1.0, 1.0)
axes[2].set_xticks([-1.0, -0.5, 0.0, 0.5, 1.0])
axes[2].set_ylim(*ENERGY_WINDOW)
axes[2].grid(alpha=0.3)

plt.show()


In [ ]:
# Standalone PBC/OBC DW Hamiltonian spectrum comparison
NX, NY = 20, 40
ALPHA_1, ALPHA_2 = 30, 1
ALPHA_OBC = 1
ENERGY_WINDOW = (-3.0, 3.0)

# Build both models independently so this cell does not depend on earlier cells
model_pbc = classA_U1FGTN(NX, NY, nshell=None, DW=True, alpha_1=ALPHA_1, alpha_2=ALPHA_2)
H_eq_pbc = model_pbc._domain_wall_hamiltonian(periodic=True)
evals_eq_pbc = np.linalg.eigvalsh(H_eq_pbc)
G_eq_pbc = model_pbc.G_CI_domain_wall(periodic=True)

model_obc = classA_U1FGTN(NX, NY, nshell=None, DW=True, alpha_1=ALPHA_OBC, alpha_2=ALPHA_OBC)
H_eq_obc = model_obc._domain_wall_hamiltonian(periodic=False)
evals_eq_obc = np.linalg.eigvalsh(H_eq_obc)
G_eq_obc = model_obc.G_CI_domain_wall(periodic=False)

neg_count_pbc = int(np.count_nonzero(evals_eq_pbc < 0.0))
pos_count_pbc = int(np.count_nonzero(evals_eq_pbc > 0.0))
spectral_gap_pbc = float(np.min(np.abs(evals_eq_pbc)))

neg_count_obc = int(np.count_nonzero(evals_eq_obc < 0.0))
pos_count_obc = int(np.count_nonzero(evals_eq_obc > 0.0))
spectral_gap_obc = float(np.min(np.abs(evals_eq_obc)))

print(f"System size: Nx={NX}, Ny={NY}")
print(f"alpha_1={ALPHA_1}, alpha_2={ALPHA_2}")
print(f"PBC DW locations: {getattr(model_pbc, 'DW_loc', None)}")
print(f"OBC DW locations: {getattr(model_obc, 'DW_loc', None)}")
print(f"H_eq_pbc shape: {H_eq_pbc.shape}, G_eq_pbc shape: {G_eq_pbc.shape}")
print(f"H_eq_obc shape: {H_eq_obc.shape}, G_eq_obc shape: {G_eq_obc.shape}")
print(
    f"PBC negative modes={neg_count_pbc}, "
    f"positive modes={pos_count_pbc}, "
    f"min |E|={spectral_gap_pbc:.6e}"
)
print(
    f"OBC negative modes={neg_count_obc}, "
    f"positive modes={pos_count_obc}, "
    f"min |E|={spectral_gap_obc:.6e}"
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), constrained_layout=True)

# Full spectrum
axes[0].plot(
    np.arange(evals_eq_obc.size),
    evals_eq_obc,
    ".",
    ms=3,
    color="red",
    alpha=0.9,
    label=rf"OBC ($\alpha = {ALPHA_OBC}$)",
)
axes[0].plot(
    np.arange(evals_eq_pbc.size),
    evals_eq_pbc,
    ".",
    ms=2,
    color="blue",
    alpha=0.4,
    label=rf"PBC ($\alpha_{{\mathrm{{top}}}} = {ALPHA_1}$, $\alpha_{{\mathrm{{triv}}}} = {ALPHA_2}$, DWs at {getattr(model_pbc, 'DW_loc', None)})",
)
axes[0].axhline(0.0, color="k", lw=1, alpha=0.5)
axes[0].set_title(rf"DW Hamiltonian eigenspectrum: OBC vs PBC ($N_x={NX}, N_y={NY}$)")
axes[0].set_xlabel("eigenvalue index")
axes[0].set_ylabel("energy")
axes[0].grid(alpha=0.3)
axes[0].legend()

# Zoomed spectrum
axes[1].plot(
    np.arange(evals_eq_obc.size),
    evals_eq_obc,
    ".",
    ms=3,
    color="red",
    alpha=0.9,
    label="OBC",
)
axes[1].plot(
    np.arange(evals_eq_pbc.size),
    evals_eq_pbc,
    ".",
    ms=2,
    color="blue",
    alpha=0.4,
    label="PBC",
)
axes[1].axhline(0.0, color="k", lw=1, alpha=0.5)
axes[1].set_title(rf"Zoomed spectrum: $E \in [{ENERGY_WINDOW[0]:.0f}, {ENERGY_WINDOW[1]:.0f}]$")
axes[1].set_xlabel("eigenvalue index")
axes[1].set_ylabel("energy")
axes[1].set_ylim(*ENERGY_WINDOW)
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.show()


## 1b. Flattened Hamiltonian from overcomplete Wannier projectors

Constructs the flattened Hamiltonian $H = \sum_R \left[(P_{A,R}^+ + P_{B,R}^+) - (P_{A,R}^- + P_{B,R}^-)\right]$ for trial orbitals $\tau_{A,B}$ chosen as eigenstates of $\sigma_X$, $\sigma_Y$, or $\sigma_Z$. Diagonalizes $H$, projects onto the negative-eigenvalue subspace to obtain $C_\mathrm{ow} = 2 P_\mathrm{occ}^\ast - I$ (same $\pm 1$ convention as the exact $G_\mathrm{eq}$), and compares the log-chord fit of the integrated entanglement contour across trial orbital choices.

In [ ]:
# Integrated-contour log-chord analysis per trial orbital and DW-truncation choice
# Mirrors the exact-G_eq analysis in cell "3. Integrated entanglement contour..."
# but substitutes C_ow built from the flattened H for each trial_orbitals choice.
#
# Flattened H construction (per trial orbital and truncation choice):
#   H = sum_R [(P_Ap + P_Bp) - (P_Am + P_Bm)], hermitized
#   diagonalize -> take negative-eigenvalue subspace U_occ
#   P_occ = U_occ @ U_occ^dagger
#   G     = P_occ.conj()
#   C_ow  = 2*G - I          <- same +/-1 convention as G_eq from G_CI_domain_wall()
#
# Depends on: model, G_eq from cell 8; helper functions from cell 6.

TRIAL_ORBITALS_LIST = ["X", "Y", "Z"]
TRUNCATION_OPTIONS = [
    (False, "untruncated"),
    (True, "DW-truncated"),
]
TRIAL_COLORS  = {"exact": "black", "X": "tab:blue", "Y": "tab:orange", "Z": "tab:green"}
TRIAL_MARKERS = {"exact": "D", "X": "o", "Y": "s", "Z": "^"}
TRUNCATION_LINESTYLES = {False: "-", True: "--"}
TRUNCATION_MARKERS = {False: "o", True: "s"}
TRUNCATION_ALPHAS = {False: 0.55, True: 0.95}

Nlayer = 2 * model.Nx * model.Ny

# ------------------------------------------------------------------
# 1. Build C_ow for each trial orbital x DW-truncation choice
# ------------------------------------------------------------------
H_flat_by_trial = {}
C_ow = {}
evals_by_trial = {}
ky_evals_by_trial = {}
ky_vals_pi_by_trial = {}
ow_run_specs = []

for trial in TRIAL_ORBITALS_LIST:
    for dw_truncation, trunc_label in TRUNCATION_OPTIONS:
        key = (trial, dw_truncation)
        model.construct_OW_projectors(
            nshell=model.nshell,
            DW=model.DW,
            trial_orbitals=trial,
            dw_truncation=dw_truncation,
        )

        # Vectorised equivalent of the per-site loop in the reference code.
        WA_p = model.WF_Ap.reshape(Nlayer, -1)
        WB_p = model.WF_Bp.reshape(Nlayer, -1)
        WA_m = model.WF_Am.reshape(Nlayer, -1)
        WB_m = model.WF_Bm.reshape(Nlayer, -1)

        H_flat = (WA_p @ WA_p.conj().T + WB_p @ WB_p.conj().T
                  - WA_m @ WA_m.conj().T - WB_m @ WB_m.conj().T)
        comm = WA_p @ WA_p.conj().T @ WA_m @ WA_m.conj().T - WA_m @ WA_m.conj().T @ WA_p @ WA_p.conj().T
        print(
            f"trial={trial}, dw_truncation={dw_truncation}: "
            f"max |[P_Ap, P_Am]| = {np.max(np.abs(comm)):.3e}"
        )
        H_flat = 0.5 * (H_flat + H_flat.conj().T)
        H_flat_by_trial[key] = H_flat

        evals, evecs = np.linalg.eigh(H_flat)
        evals_by_trial[key] = evals

        H_flat6 = unflatten_to_G6(H_flat, model.Nx, model.Ny, top=True)
        ky_vals_trial, ky_blocks_trial = _fft_y_blocks_from_flat(H_flat6, norm="ortho", hermitize=True)
        ky_evals_trial = np.empty((model.Ny, 2 * model.Nx), dtype=float)
        for j, B in enumerate(ky_blocks_trial):
            ky_evals_trial[j, :] = np.sort(np.real_if_close(np.linalg.eigvalsh(B)))
        ky_vals_trial = np.fft.fftshift(ky_vals_trial) / np.pi
        ky_evals_trial = np.fft.fftshift(ky_evals_trial, axes=0)
        ky_vals_pi_by_trial[key] = ky_vals_trial
        ky_evals_by_trial[key] = ky_evals_trial

        occ_mask = evals < 0.0
        if not np.any(occ_mask):
            occ_mask = np.zeros_like(evals, dtype=bool)
            occ_mask[np.argsort(evals)[: Nlayer // 2]] = True
        U_occ = evecs[:, occ_mask]
        P_occ = U_occ @ U_occ.conj().T
        G_ow  = P_occ.conj()
        C_ow[key] = 2.0 * G_ow - np.eye(Nlayer, dtype=np.complex128)
        C_ow[f"{trial}_{'truncated' if dw_truncation else 'untruncated'}"] = C_ow[key]

        ow_run_specs.append(
            {
                "key": key,
                "trial": trial,
                "dw_truncation": dw_truncation,
                "trunc_label": trunc_label,
                "label": rf"$\sigma_{{{trial}}}$, {trunc_label}",
                "color": TRIAL_COLORS[trial],
                "marker": TRUNCATION_MARKERS[dw_truncation],
                "linestyle": TRUNCATION_LINESTYLES[dw_truncation],
                "alpha": TRUNCATION_ALPHAS[dw_truncation],
                "G": C_ow[key],
            }
        )

# Backward-compatible aliases for later notebook cells. Because this edited cell was
# requested from the dw_truncation=True version, keep the legacy trial names pointing
# at the DW-truncated covariances.
for trial in TRIAL_ORBITALS_LIST:
    C_ow[trial] = C_ow[(trial, True)]

exact_run_spec = {
    "key": "exact",
    "trial": "exact",
    "dw_truncation": None,
    "trunc_label": "exact",
    "label": "exact",
    "color": TRIAL_COLORS["exact"],
    "marker": TRIAL_MARKERS["exact"],
    "linestyle": "-",
    "alpha": 1.0,
    "G": G_eq,
}
all_runs = [exact_run_spec] + ow_run_specs

# ------------------------------------------------------------------
# 2. Eigenspectrum of flattened H for each trial orbital
#    Each subplot overlays dw_truncation=False and dw_truncation=True.
# ------------------------------------------------------------------
fig, axes = plt.subplots(2, len(TRIAL_ORBITALS_LIST), figsize=(5.5 * len(TRIAL_ORBITALS_LIST), 8.0),
                         constrained_layout=True)
for col, trial in enumerate(TRIAL_ORBITALS_LIST):
    ax = axes[0, col]
    for dw_truncation, trunc_label in TRUNCATION_OPTIONS:
        key = (trial, dw_truncation)
        ax.plot(
            evals_by_trial[key],
            marker=TRUNCATION_MARKERS[dw_truncation],
            ls="none",
            ms=2.5,
            color=TRIAL_COLORS[trial],
            alpha=TRUNCATION_ALPHAS[dw_truncation],
            label=trunc_label,
        )
    ax.set_title(rf"Flattened $H$,  $\tau \in \sigma_{{{trial}}}$")
    ax.set_xlabel("eigenvalue index")
    ax.set_ylabel("energy")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

    ax = axes[1, col]
    for dw_truncation, trunc_label in TRUNCATION_OPTIONS:
        key = (trial, dw_truncation)
        for i_band, band in enumerate(ky_evals_by_trial[key].T):
            ax.plot(
                ky_vals_pi_by_trial[key],
                band,
                marker=TRUNCATION_MARKERS[dw_truncation],
                ls="none",
                ms=2.5,
                color=TRIAL_COLORS[trial],
                alpha=0.35 if not dw_truncation else 0.7,
                label=trunc_label if i_band == 0 else None,
            )
    ax.set_title(rf"Flattened $H$ vs $k_y/\pi$,  $\tau \in \sigma_{{{trial}}}$")
    ax.set_xlabel(r"$k_y/\pi$")
    ax.set_ylabel("energy")
    ax.set_xlim(-1.0, 1.0)
    ax.set_xticks([-1.0, -0.5, 0.0, 0.5, 1.0])
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.suptitle("Flattened Hamiltonian spectra per trial orbital: DW truncation comparison")
plt.show()

fig, axes = plt.subplots(1, len(TRIAL_ORBITALS_LIST), figsize=(5.5 * len(TRIAL_ORBITALS_LIST), 4.0),
                         constrained_layout=True)
for ax, trial in zip(axes, TRIAL_ORBITALS_LIST):
    for dw_truncation, trunc_label in TRUNCATION_OPTIONS:
        key = (trial, dw_truncation)
        ax.hist(
            evals_by_trial[key],
            bins=60,
            histtype="stepfilled" if not dw_truncation else "step",
            linewidth=1.6 if dw_truncation else 1.0,
            color=TRIAL_COLORS[trial],
            alpha=0.35 if not dw_truncation else 0.95,
            label=trunc_label,
        )
    ax.set_title(rf"Flattened $H$ eigenvalue histogram,  $\tau \in \sigma_{{{trial}}}$")
    ax.set_xlabel("energy")
    ax.set_ylabel("count")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.suptitle("Flattened Hamiltonian eigenvalue histograms: DW truncation comparison")
plt.show()

# Print mean(abs(spec(H))) for each flattened-H trial orbital and truncation choice.
for trial in TRIAL_ORBITALS_LIST:
    for dw_truncation, trunc_label in TRUNCATION_OPTIONS:
        key = (trial, dw_truncation)
        mean_abs_spec = np.mean(np.abs(evals_by_trial[key]))
        print(
            f"Trial orbital {trial}, dw_truncation={dw_truncation}: "
            f"mean(abs(spec(H_flat))) = {mean_abs_spec:.6f} "
            f"computed from {len(evals_by_trial[key])} eigenvalues."
        )


# ------------------------------------------------------------------
# 3. Compute integrated-contour curves for every run x x-interval
# ------------------------------------------------------------------
x0 = int(model.DW_loc[0]) % model.Nx
x1 = int(model.DW_loc[1]) % model.Nx
X_INTERVALS = [
    (0, model.Nx - 1),
    (x0 - 1, x0 + 1),
    (x1 - 1, x1 + 1),
]

Ay_values = np.arange(2, model.Ny - 1, dtype=int)
x_full    = np.arange(model.Nx, dtype=int)

results = {run["key"]: {} for run in all_runs}
for run in all_runs:
    run_key = run["key"]
    C_run = run["G"]
    for x_start, x_stop_inclusive in X_INTERVALS:
        x_interval = contiguous_x_block(model.Nx, x_start, x_stop_inclusive)
        curve = np.zeros_like(Ay_values, dtype=float)
        for i, Ay in enumerate(Ay_values):
            y_region = np.arange(Ay, dtype=int)
            _, s_map = contour_for_subsystem(model, C_run, x_full, y_region)
            curve[i] = float(np.sum(s_map[x_interval, :]))
        results[run_key][(x_start, x_stop_inclusive)] = {"x_interval": x_interval, "curve": curve}


def _fit_line_with_error(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if x.size < 3:
        return {"m": np.nan, "m_err": np.nan, "b": np.nan, "r2": np.nan,
                "x": x, "y": y, "y_fit": np.full_like(y, np.nan)}
    try:
        coeffs, cov = np.polyfit(x, y, 1, cov=True)
        m, b = float(coeffs[0]), float(coeffs[1])
        m_err = float(np.sqrt(cov[0, 0])) if np.isfinite(cov[0, 0]) else np.nan
    except Exception:
        m, b, m_err = np.nan, np.nan, np.nan
    if not np.isfinite(m):
        return {"m": np.nan, "m_err": np.nan, "b": np.nan, "r2": np.nan,
                "x": x, "y": y, "y_fit": np.full_like(y, np.nan)}
    y_fit = m * x + b
    ss_res = float(np.sum((y - y_fit) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = np.nan if ss_tot == 0 else (1.0 - ss_res / ss_tot)
    return {"m": m, "m_err": m_err, "b": b, "r2": r2, "x": x, "y": y, "y_fit": y_fit}


# ------------------------------------------------------------------
# 4. Figure 1: raw curves vs Ay
#    Each trial-orbital subplot overlays exact, untruncated, and DW-truncated curves.
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, len(TRIAL_ORBITALS_LIST), figsize=(6.2 * len(TRIAL_ORBITALS_LIST), 4.8),
                         constrained_layout=True)
interval_colors = plt.cm.viridis(np.linspace(0.08, 0.92, len(X_INTERVALS)))
dw_loc = getattr(model, "DW_loc", None)

for ax, trial in zip(axes, TRIAL_ORBITALS_LIST):
    for color, interval in zip(interval_colors, X_INTERVALS):
        x_interval = results["exact"][interval]["x_interval"]
        interval_label = rf"$x \in [{int(x_interval[0])}, {int(x_interval[-1])}]$"

        exact_curve = results["exact"][interval]["curve"]
        ax.plot(
            Ay_values,
            exact_curve,
            marker=TRIAL_MARKERS["exact"],
            ms=2.5,
            lw=1.2,
            ls=":",
            color=color,
            alpha=0.8,
            label=interval_label + ", exact",
        )

        for dw_truncation, trunc_label in TRUNCATION_OPTIONS:
            key = (trial, dw_truncation)
            curve = results[key][interval]["curve"]
            ax.plot(
                Ay_values,
                curve,
                marker=TRUNCATION_MARKERS[dw_truncation],
                ms=3,
                lw=1.5,
                ls=TRUNCATION_LINESTYLES[dw_truncation],
                color=color,
                alpha=TRUNCATION_ALPHAS[dw_truncation],
                label=interval_label + f", {trunc_label}",
            )

    ax.set_title(rf"Integrated contour, $\tau \in \sigma_{{{trial}}}$")
    ax.set_xlabel(r"$A_y$")
    ax.set_ylabel(r"$\sum_{x \in I_x}\sum_{y \in A} s_A(x,y)$")
    ax.grid(alpha=0.3)
    ax.legend(title="x interval / run", fontsize=6.5, ncol=1)

textbox = (
    rf"$N_x={model.Nx},\ N_y={model.Ny}$" "\n"
    rf"$\alpha_{{\mathrm{{triv}}}}={model.alpha_1},\ \alpha_{{\mathrm{{top}}}}={model.alpha_2}$" "\n"
    rf"DWs at $x={dw_loc}$"
)
axes[0].text(0.03, 0.55, textbox, transform=axes[0].transAxes, ha="left", va="top",
             fontsize=9, linespacing=1.4,
             bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.85))
plt.suptitle(r"Integrated contour vs $A_y$ -- exact vs OW flattened $H$, with/without DW truncation")
plt.show()

# ------------------------------------------------------------------
# 5. Figure 2: log-chord fit (one panel per x-interval, all runs overlaid)
# ------------------------------------------------------------------
mid = model.Ny / 2
mask_lower = Ay_values < mid
mask_upper = Ay_values > mid
Ay_lower = Ay_values[mask_lower]
Ay_upper = Ay_values[mask_upper]
x_lower  = np.log(np.sin(np.pi * Ay_lower / model.Ny))
x_upper  = np.log(np.sin(np.pi * Ay_upper / model.Ny))

fit_results = {run["key"]: {} for run in all_runs}

fig, axes = plt.subplots(1, len(X_INTERVALS), figsize=(7.2 * len(X_INTERVALS), 5.4),
                         constrained_layout=True)

for ax, interval in zip(axes, X_INTERVALS):
    x_interval = contiguous_x_block(model.Nx, interval[0], interval[1])
    ax.set_title(rf"$x \in [{int(x_interval[0])},\,{int(x_interval[-1])}]$")
    ax.set_xlabel(r"$\log\!\left[\sin\!\left(\pi A_y/N_y\right)\right]$")
    ax.set_ylabel(r"$\sum_{x \in I_x}\sum_{y \in A} s_A(x,y)$")
    ax.grid(alpha=0.3)

    for run in all_runs:
        run_key = run["key"]
        curve = results[run_key][interval]["curve"]
        lower_fit = _fit_line_with_error(x_lower, curve[mask_lower])
        upper_fit = _fit_line_with_error(x_upper, curve[mask_upper])
        fit_results[run_key][interval] = {"lower": lower_fit, "upper": upper_fit}

        run_label = run["label"]
        color = run["color"]
        marker = run["marker"]
        linestyle = run["linestyle"]
        alpha = run["alpha"]

        ax.plot(lower_fit["x"], lower_fit["y"], marker, ms=3, color=color, alpha=alpha,
                label=(run_label
                       + rf" lower: $m={lower_fit['m']:.4f}\pm{lower_fit['m_err']:.2e}$"
                       + rf", $R^2={lower_fit['r2']:.4f}$"))
        ax.plot(lower_fit["x"], lower_fit["y_fit"], linestyle, lw=1.4, color=color, alpha=alpha)

        ax.plot(upper_fit["x"], upper_fit["y"], marker, ms=3, color=color, alpha=0.35 * alpha,
                label=(run_label
                       + rf" upper: $m={upper_fit['m']:.4f}\pm{upper_fit['m_err']:.2e}$"
                       + rf", $R^2={upper_fit['r2']:.4f}$"))
        ax.plot(upper_fit["x"], upper_fit["y_fit"], linestyle, lw=1.4, color=color, alpha=0.45 * alpha)

    ax.legend(fontsize=5.8, ncol=1)

plt.suptitle(r"Log-chord fit: exact $G_{\mathrm{eq}}$ vs OW flattened $H$ -- DW truncation comparison")
plt.show()

# ------------------------------------------------------------------
# 6. Print fit summary
# ------------------------------------------------------------------
print("Fit summary")
print("===========")
for interval in X_INTERVALS:
    x_interval = contiguous_x_block(model.Nx, interval[0], interval[1])
    print(f"x interval {interval}  ->  x = [{int(x_interval[0])}, ..., {int(x_interval[-1])}]")
    print(f"  {'run':<24}  {'branch':<7}  {'slope':>10}  {'slope_err':>12}  {'intercept':>12}  {'R2':>8}")
    print(f"  {'-'*24}  {'-'*7}  {'-'*10}  {'-'*12}  {'-'*12}  {'-'*8}")
    for run in all_runs:
        for branch in ("lower", "upper"):
            r = fit_results[run["key"]][interval][branch]
            print(
                f"  {run['label']:<24}  {branch:<7}  {r['m']:>+10.6f}  "
                f"{r['m_err']:>12.4e}  {r['b']:>12.6f}  {r['r2']:>8.6f}"
            )

print(f"A_y scan = {Ay_values[0]} ... {Ay_values[-1]}")

In [ ]:
# Current maps from the matching exact / OW-flattened Hamiltonians and covariances

G_flat_X = C_ow["X"]
G_flat_Y = C_ow["Y"]
G_flat_Z = C_ow["Z"]

Nlayer = 2 * model.Nx * model.Ny

H_flat_by_trial = {}
for trial in TRIAL_ORBITALS_LIST:
    model.construct_OW_projectors(nshell=model.nshell, DW=model.DW, trial_orbitals=trial)
    WA_p = model.WF_Ap.reshape(Nlayer, -1)
    WB_p = model.WF_Bp.reshape(Nlayer, -1)
    WA_m = model.WF_Am.reshape(Nlayer, -1)
    WB_m = model.WF_Bm.reshape(Nlayer, -1)
    H_trial = (
        WA_p @ WA_p.conj().T + WB_p @ WB_p.conj().T
        - WA_m @ WA_m.conj().T - WB_m @ WB_m.conj().T
    )
    H_flat_by_trial[trial] = 0.5 * (H_trial + H_trial.conj().T)

H_flat_X = H_flat_by_trial["X"]
H_flat_Y = H_flat_by_trial["Y"]
H_flat_Z = H_flat_by_trial["Z"]

G_by_state = {
    "exact": G_eq,
    "X": G_flat_X,
    "Y": G_flat_Y,
    "Z": G_flat_Z,
}

H_by_state = {
    "exact": H_eq,
    "X": H_flat_X,
    "Y": H_flat_Y,
    "Z": H_flat_Z,
}

# Switch between one shared Hamiltonian for all current maps and the Hamiltonian
# naturally associated with each covariance.
CURRENT_HAMILTONIAN_MODE = "matched"  # choices: "matched", "shared_H_eq"

if CURRENT_HAMILTONIAN_MODE == "matched":
    state_map = {
        name: (H_by_state[name], G_by_state[name])
        for name in ["exact", "X", "Y", "Z"]
    }
    current_hamiltonian_label = r"matched $H$"
elif CURRENT_HAMILTONIAN_MODE == "shared_H_eq":
    state_map = {
        name: (H_eq, G_by_state[name])
        for name in ["exact", "X", "Y", "Z"]
    }
    current_hamiltonian_label = r"shared $H_{\mathrm{eq}}$"
else:
    raise ValueError(
        f"CURRENT_HAMILTONIAN_MODE must be 'matched' or 'shared_H_eq'; got {CURRENT_HAMILTONIAN_MODE!r}"
    )

# Verify that current_maps_from_hamiltonian reproduces the generic quadratic-fermion
# bond-current formula in the same real-space DW Chern-insulator gauge used by the
# OW construction.
sigma_x = np.array([[0, 1], [1, 0]], dtype=np.complex128)
sigma_y = np.array([[0, -1j], [1j, 0]], dtype=np.complex128)
sigma_z = np.array([[1, 0], [0, -1]], dtype=np.complex128)
hop_x = -0.5 * sigma_z - 0.5j * sigma_x
hop_y = -0.5 * sigma_z - 0.5j * sigma_y

C_eq = 0.5 * (G_eq + np.eye(Nlayer, dtype=np.complex128))
C_eq6 = np.transpose(
    C_eq.reshape(2, model.Nx, model.Ny, 2, model.Nx, model.Ny, order="F"),
    (1, 2, 0, 4, 5, 3),
)

x_idx = np.arange(model.Nx)[:, None]
y_idx = np.arange(model.Ny)[None, :]
x_next = (x_idx + 1) % model.Nx
y_next = (y_idx + 1) % model.Ny

H_eq6 = np.transpose(
    H_eq.reshape(2, model.Nx, model.Ny, 2, model.Nx, model.Ny, order="F"),
    (1, 2, 0, 4, 5, 3),
)

H_x_ref = H_eq6[x_idx, y_idx, :, x_next, y_idx, :]
H_y_ref = H_eq6[x_idx, y_idx, :, x_idx, y_next, :]
C_x_ref = C_eq6[x_next, y_idx, :, x_idx, y_idx, :]
C_y_ref = C_eq6[x_idx, y_next, :, x_idx, y_idx, :]
J_x_ref = 2.0 * np.imag(np.einsum("xyab,xyba->xy", H_x_ref, C_x_ref, optimize=True))
J_y_ref = 2.0 * np.imag(np.einsum("xyab,xyba->xy", H_y_ref, C_y_ref, optimize=True))

J_x_exact, J_y_exact = model.current_maps_from_hamiltonian(H_eq, G_eq)
print("Gauge consistency check against DW real-space hopping convention")
print(f"  hop_x =\n{hop_x}")
print(f"  hop_y =\n{hop_y}")
print(f"  max |J_x - J_x_ref| = {np.max(np.abs(J_x_exact - J_x_ref)):.3e}")
print(f"  max |J_y - J_y_ref| = {np.max(np.abs(J_y_exact - J_y_ref)):.3e}")
print(f"Current-map Hamiltonian mode: {CURRENT_HAMILTONIAN_MODE}")

current_maps = {
    name: model.current_maps_from_hamiltonian(H_state, G_state)
    for name, (H_state, G_state) in state_map.items()
}

shared_absmax = max(
    max(float(np.max(np.abs(J_x))), float(np.max(np.abs(J_y))))
    for J_x, J_y in current_maps.values()
)

plot_titles = {
    "exact": rf"Exact $G_{{\mathrm{{eq}}}}$ current maps ({current_hamiltonian_label})",
    "X": rf"OW flattened $H$, $\tau \in \sigma_{{X}}$ current maps ({current_hamiltonian_label})",
    "Y": rf"OW flattened $H$, $\tau \in \sigma_{{Y}}$ current maps ({current_hamiltonian_label})",
    "Z": rf"OW flattened $H$, $\tau \in \sigma_{{Z}}$ current maps ({current_hamiltonian_label})",
}

for name in ["exact", "X", "Y", "Z"]:
    J_x, J_y = current_maps[name]
    fig, axes = model.plot_current_maps(
        J_x,
        J_y,
        figsize=(18, 5),
        cmap="RdBu_r",
        quiver_cmap="plasma",
        vmin=-shared_absmax,
        vmax=shared_absmax,
    )
    fig.suptitle(plot_titles[name], y=1.02)
    plt.show()


## 2. Plot the half-system entanglement contour heatmap and local Chern marker heatmaps

The half-system cut below keeps all `x` sites and the first `Ny // 2` columns in `y`. Because the DW construction is translation-invariant along `y`, you can shift `HALF_Y_START` without changing the equilibrium answer.

In [ ]:
HALF_Y_START = 0
x_full = np.arange(model.Nx, dtype=int)
y_half = contiguous_y_block(model.Ny, model.Ny // 2, start=HALF_Y_START)

G_half, s_half = contour_for_subsystem(model, G_eq, x_full, y_half)
chern_tanh = model.local_chern_marker_flat(G_eq, apply_tanh=True)

dw_loc = getattr(model, "DW_loc", None)
ec_textbox = (
    rf"$N_x={model.Nx},\ N_y={model.Ny}$" "\n"
    rf"$\alpha_{{\mathrm{{triv}}}}={model.alpha_1},\ \alpha_{{\mathrm{{top}}}}={model.alpha_2}$" "\n"
    rf"DWs at $x={dw_loc}$"
)

# ---------------------------
# Figure 1: EC heatmap + EC y-profiles
# ---------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), constrained_layout=True)

im0 = axes[0].imshow(s_half.T, origin="lower", cmap="Blues", aspect="auto")
axes[0].set_title(r"Half-system entanglement contour $s_A(x,y)$")
axes[0].set_xlabel("x")
axes[0].set_ylabel("subsystem y")
fig.colorbar(im0, ax=axes[0], label="contour weight")

axes[0].text(
    1.0,
    1.0,
    ec_textbox,
    transform=axes[0].transAxes,
    ha="right",
    va="top",
    fontsize=8,
    linespacing=1.4,
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.85),
)

# Integer ticks every 2, plus unit-cell gridlines on the EC heatmap
ny_plot, nx_plot = s_half.T.shape
axes[0].set_xticks(np.arange(0, nx_plot, 2))
axes[0].set_yticks(np.arange(0, ny_plot, 2))
axes[0].set_xticks(np.arange(-0.5, nx_plot, 1), minor=True)
axes[0].set_yticks(np.arange(-0.5, ny_plot, 1), minor=True)
axes[0].grid(which="minor", color="white", linestyle="-", linewidth=0.5, alpha=0.35)
axes[0].tick_params(which="minor", bottom=False, left=False)

# Entanglement-contour cuts vs y for fixed x values in [0, Nx//2)
x_list = np.arange(0, model.Nx // 2, dtype=int)
colors = plt.cm.viridis(np.linspace(0.05, 0.95, len(x_list)))
y_vals = np.arange(s_half.shape[1], dtype=int)

for x, color in zip(x_list, colors):
    curve = s_half[x, :]
    y_plot = y_vals[: len(curve) // 2]
    curve_plot = curve[: len(curve) // 2]

    axes[1].plot(
        y_plot,
        curve_plot,
        marker="o",
        ms=2.8,
        lw=1.4,
        ls="-",
        color=color,
    )

    # label each curve directly near its last positive point in the plotted range
    positive_idx = np.where(curve_plot > 0)[0]
    if positive_idx.size > 0:
        j = positive_idx[-1]
        x_text = y_plot[j]
        y_text = curve_plot[j]
    else:
        j = len(curve_plot) - 1
        x_text = y_plot[j]
        y_text = max(curve_plot[j], 1e-16)

    axes[1].text(
        x_text + 0.1,
        y_text,
        f"x={x}",
        color=color,
        fontsize=8,
        ha="left",
        va="center",
    )

axes[1].set_title(r"Entanglement contour profiles vs $y$")
axes[1].set_xlabel("subsystem y")
axes[1].set_ylabel(r"$s_A(x,y)$")
axes[1].set_xticks(np.arange(0, s_half.shape[1] // 2, 2))
axes[1].grid(alpha=0.3)
axes[1].set_yscale("log")

plt.show()

# ---------------------------
# Figure 2: Local Chern marker
# ---------------------------
fig, ax = plt.subplots(1, 1, figsize=(6.2, 4.8), constrained_layout=True)

im = ax.imshow(chern_tanh.T, origin="lower", cmap="RdBu_r", aspect="auto", vmin=-1.0, vmax=1.0)
ax.set_title("Local Chern marker (tanh)")
ax.set_xlabel("x")
ax.set_ylabel("y")
fig.colorbar(im, ax=ax, label="tanh(C(r))")

ny_plot, nx_plot = chern_tanh.T.shape
ax.set_xticks(np.arange(0, nx_plot, 2))
ax.set_yticks(np.arange(0, ny_plot, 2))
ax.set_xticks(np.arange(-0.5, nx_plot, 1), minor=True)
ax.set_yticks(np.arange(-0.5, ny_plot, 1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=0.5, alpha=0.35)
ax.tick_params(which="minor", bottom=False, left=False)

chern_textbox = (
    rf"$N_x={model.Nx},\ N_y={model.Ny}$" "\n"
    rf"$\alpha_{{\mathrm{{triv}}}}={model.alpha_1},\ \alpha_{{\mathrm{{top}}}}={model.alpha_2}$" "\n"
    rf"DWs at $x={dw_loc}$"
)

ax.text(
    1.0,
    1.0,
    chern_textbox,
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=8,
    linespacing=1.4,
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.85),
)

plt.show()


In [ ]:
x_full = np.arange(model.Nx, dtype=int)
y_half = contiguous_y_block(model.Ny, model.Ny // 2, start=HALF_Y_START)
idx_half = subsystem_indices(model.Nx, model.Ny, x_full, y_half)


def modular_hamiltonian_from_covariance(G_sub, eps=1e-12):
    G_sub = np.asarray(G_sub, dtype=np.complex128)
    evals, evecs = np.linalg.eigh(G_sub)
    evals = np.clip(np.real_if_close(evals), -1.0 + eps, 1.0 - eps)
    h_eigs = -2.0 * np.arctanh(evals)
    H_mod = (evecs * h_eigs[None, :]) @ evecs.conj().T
    return 0.5 * (H_mod + H_mod.conj().T)


G_half_by_state_spec = {
    "exact": reduced_covariance(G_eq, idx_half),
    "X": reduced_covariance(G_flat_X, idx_half),
    "Y": reduced_covariance(G_flat_Y, idx_half),
    "Z": reduced_covariance(G_flat_Z, idx_half),
}

C_half_by_state = {}
occ_half_by_state = {}
modular_energies_unsorted_by_state = {}
modular_energies_by_state = {}
H_mod_half_by_state = {}

for name, G_half_state in G_half_by_state_spec.items():
    I_half = np.eye(G_half_state.shape[0], dtype=np.complex128)
    C_half_state = 0.5 * (I_half + G_half_state)
    C_half_state = 0.5 * (C_half_state + C_half_state.conj().T)
    occ_half_state = np.linalg.eigvalsh(C_half_state)
    occ_half_state = np.clip(np.real_if_close(occ_half_state), 1e-12, 1.0 - 1e-12)
    modular_energies_unsorted_state = np.real(np.log((1.0 - occ_half_state) / occ_half_state))
    H_mod_half_by_state[name] = modular_hamiltonian_from_covariance(G_half_state)
    C_half_by_state[name] = C_half_state
    occ_half_by_state[name] = occ_half_state
    modular_energies_unsorted_by_state[name] = modular_energies_unsorted_state
    modular_energies_by_state[name] = np.sort(modular_energies_unsorted_state)

# Preserve the exact-state aliases used elsewhere in the notebook.
G_half_mod = G_half_by_state_spec["exact"]
C_half = C_half_by_state["exact"]
occ_half = occ_half_by_state["exact"]
modular_energies_unsorted = modular_energies_unsorted_by_state["exact"]
H_mod_half = H_mod_half_by_state["exact"]
modular_energies = modular_energies_by_state["exact"]

all_modular_energies = np.concatenate([modular_energies_by_state[name] for name in ["exact", "X", "Y", "Z"]])
shared_ylim = (float(np.min(all_modular_energies)), float(np.max(all_modular_energies)))
x_spec = np.arange(modular_energies.size, dtype=int)
spec_textbox = (
    f"A = [0, {model.Nx}) x [{int(y_half[0])}, {int(y_half[-1]) + 1})" "\n"
    f"Nx={model.Nx}, Ny={model.Ny}" "\n"
    f"DWs at x={dw_loc}"
)

plot_titles_mod_spec = {
    "exact": r"Exact $G_{\mathrm{eq}}$",
    "X": r"OW $G_{\mathrm{flat},X}$",
    "Y": r"OW $G_{\mathrm{flat},Y}$",
    "Z": r"OW $G_{\mathrm{flat},Z}$",
}

fig, axes = plt.subplots(2, 2, figsize=(14.0, 9.0), constrained_layout=True)
for ax, name in zip(axes.ravel(), ["exact", "X", "Y", "Z"]):
    ax.plot(x_spec, modular_energies_by_state[name], ".", ms=3, color=TRIAL_COLORS.get(name, "tab:blue"))
    ax.axhline(0.0, color="k", lw=1, alpha=0.5)
    ax.grid(alpha=0.3)
    ax.set_xlabel("entanglement-mode index")
    ax.set_ylabel("entanglement energy")
    ax.set_ylim(*shared_ylim)
    ax.set_title(plot_titles_mod_spec[name])

axes[0, 0].text(
    0.03,
    0.97,
    spec_textbox,
    transform=axes[0, 0].transAxes,
    ha="left",
    va="top",
    fontsize=9,
    linespacing=1.4,
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.85),
)
fig.suptitle(r"Half-system modular Hamiltonian spectra", y=1.02)

plt.show()

print(
    f"Half-system cut A = [0, {model.Nx}) x [{int(y_half[0])}, {int(y_half[-1]) + 1}) | "
    f"subsystem dimension = {G_half_mod.shape[0]}"
)
for name in ["exact", "X", "Y", "Z"]:
    print(
        f"  {name}: H_mod shape = {H_mod_half_by_state[name].shape} | "
        f"C_A eig range = [{occ_half_by_state[name].min():.3e}, {occ_half_by_state[name].max():.3e}] | "
        f"energy range = [{modular_energies_by_state[name][0]:.6g}, {modular_energies_by_state[name][-1]:.6g}] | "
        f"number of entanglement modes = {modular_energies_by_state[name].size}"
    )


In [ ]:
# Half-system modular-current maps for the exact and OW-flattened covariances

x_full = np.arange(model.Nx, dtype=int)
y_half = contiguous_y_block(model.Ny, model.Ny // 2, start=HALF_Y_START)
idx_half = subsystem_indices(model.Nx, model.Ny, x_full, y_half)
Ny_sub = len(y_half)


def modular_hamiltonian_from_covariance(G_sub, eps=1e-12):
    G_sub = np.asarray(G_sub, dtype=np.complex128)
    evals, evecs = np.linalg.eigh(G_sub)
    evals = np.clip(np.real_if_close(evals), -1.0 + eps, 1.0 - eps)
    h_eigs = -2.0 * np.arctanh(evals)
    H_mod = (evecs * h_eigs[None, :]) @ evecs.conj().T
    return 0.5 * (H_mod + H_mod.conj().T)


G_half_by_state = {
    "exact": reduced_covariance(G_eq, idx_half),
    "X": reduced_covariance(G_flat_X, idx_half),
    "Y": reduced_covariance(G_flat_Y, idx_half),
    "Z": reduced_covariance(G_flat_Z, idx_half),
}

H_mod_by_state = {
    name: modular_hamiltonian_from_covariance(G_sub)
    for name, G_sub in G_half_by_state.items()
}

modular_current_maps = {
    name: model.current_maps_from_hamiltonian(H_mod_by_state[name], G_half_by_state[name], Nx=model.Nx, Ny=Ny_sub)
    for name in ["exact", "X", "Y", "Z"]
}

shared_absmax_mod = max(
    max(float(np.max(np.abs(J_x))), float(np.max(np.abs(J_y))))
    for J_x, J_y in modular_current_maps.values()
)

plot_titles_mod = {
    "exact": r"Half-system modular current maps from exact $G_{\mathrm{eq}}$",
    "X": r"Half-system modular current maps from OW $G_{\mathrm{flat},X}$",
    "Y": r"Half-system modular current maps from OW $G_{\mathrm{flat},Y}$",
    "Z": r"Half-system modular current maps from OW $G_{\mathrm{flat},Z}$",
}

print(
    f"Half-system modular current cut A = [0, {model.Nx}) x [{int(y_half[0])}, {int(y_half[-1]) + 1}) | "
    f"subsystem dimension = {G_half_by_state['exact'].shape[0]}"
)
for name in ["exact", "X", "Y", "Z"]:
    h_mod_eigs = np.linalg.eigvalsh(H_mod_by_state[name])
    J_x, J_y = modular_current_maps[name]
    print(
        f"  {name}: H_mod eig range = [{h_mod_eigs[0]:.6g}, {h_mod_eigs[-1]:.6g}] | "
        f"max |J_x| = {np.max(np.abs(J_x)):.3e}, max |J_y| = {np.max(np.abs(J_y)):.3e}"
    )

for name in ["exact", "X", "Y", "Z"]:
    J_x, J_y = modular_current_maps[name]
    fig, axes = model.plot_current_maps(
        J_x,
        J_y,
        figsize=(18, 5),
        cmap="RdBu_r",
        quiver_cmap="plasma",
        vmin=-shared_absmax_mod,
        vmax=shared_absmax_mod,
    )
    fig.suptitle(plot_titles_mod[name], y=1.02)
    plt.show()


## 2b. Squared correlation profiles along `y` for fixed `x` in the equilibrium DW GS

This cell evaluates the fixed-`x` squared correlation profile $C_G(x_0, r_y)$ for the exact domain-wall ground state.

- `r_y` runs over `0, ..., Ny//2 - 1`
- `x_0` runs over `0, ..., Nx//2 - 1`
- curves are colored by `x_0` and labeled explicitly in the legend


In [ ]:
# Squared correlation profiles vs y for the equilibrium DW GS
Nlayer = model.Ntot // 2
G_eq_arr = np.asarray(G_eq, dtype=np.complex128)
if G_eq_arr.shape == (model.Ntot, model.Ntot):
    G_top = G_eq_arr[:Nlayer, :Nlayer]
elif G_eq_arr.shape == (Nlayer, Nlayer):
    G_top = G_eq_arr
else:
    raise ValueError(f'Unexpected G_eq shape: {G_eq_arr.shape}')

G2 = 0.5 * (G_top + np.eye(Nlayer, dtype=np.complex128))
G6 = G2.reshape(2, NX, NY, 2, NX, NY, order='F')
Gker = np.transpose(G6, (1, 2, 0, 4, 5, 3))  # (x,y,2,x',y',2)

FIT_RY_MIN = 3
FIT_RY_MAX = min(15, NY // 2 - 1)

ry_vals_full = np.arange(1, NY // 2, dtype=int)
ry_vals_fit = np.arange(1, NY // 2, dtype=int)
x_vals = np.arange(NX//2, dtype=int)
dw_loc = np.asarray(getattr(model, 'DW_loc', []), dtype=int) % NX
dw_x_vals = np.sort(np.array([dw_loc[0]-1, dw_loc[0], dw_loc[0]+1, dw_loc[1]-1, dw_loc[1], dw_loc[1]+1], dtype=int) % NX)
if dw_x_vals.size == 0:
    raise ValueError('Expected nonempty DW_loc for the domain-wall benchmark.')


def squared_corr_profile(Gker_in, x0, ry_vals_arr):
    x0 = int(x0) % NX
    ry_arr = np.atleast_1d(ry_vals_arr).astype(int)
    Gx = Gker_in[x0, :, :, x0, :, :]  # (Ny,2,Ny,2)
    Y = np.arange(NY, dtype=np.intp)[:, None]
    Yp = (Y + ry_arr[None, :]) % NY
    Gx_re = np.transpose(Gx, (0, 2, 1, 3)).reshape(NY * NY, 2, 2)
    flat_ix = (Y * NY + Yp).reshape(-1)
    blocks = Gx_re[flat_ix].reshape(NY, ry_arr.size, 2, 2)
    return np.sum(np.abs(blocks) ** 2, axis=(0, 2, 3)) / (2.0 * NY)


corr_by_x = {int(x0): squared_corr_profile(Gker, x0, ry_vals_full).real for x0 in x_vals}
dw_corr_by_x = {int(x0): squared_corr_profile(Gker, x0, ry_vals_fit).real for x0 in dw_x_vals}

log_ry_full = np.log(ry_vals_full.astype(float))
chord_vals = (NY / np.pi) * np.sin(np.pi * ry_vals_fit / NY)
log_chord_vals = np.log(chord_vals)
fit_mask = (ry_vals_fit >= FIT_RY_MIN) & (ry_vals_fit <= FIT_RY_MAX)
if np.count_nonzero(fit_mask) < 2:
    raise ValueError('Need at least two points in the fit window.')

fit_results = {}
for x0 in dw_x_vals:
    C_fit = np.clip(dw_corr_by_x[int(x0)][fit_mask], 1e-300, None)
    logC_fit = np.log(C_fit)
    slope, intercept = np.polyfit(log_chord_vals[fit_mask], logC_fit, 1)
    logC_hat = slope * log_chord_vals[fit_mask] + intercept
    ss_res = float(np.sum((logC_fit - logC_hat) ** 2))
    ss_tot = float(np.sum((logC_fit - np.mean(logC_fit)) ** 2))
    r2 = np.nan if ss_tot == 0.0 else 1.0 - ss_res / ss_tot
    fit_results[int(x0)] = {
        'slope': float(slope),
        'intercept': float(intercept),
        'r2': float(r2),
        'fit_line': slope * log_chord_vals + intercept,
    }

fig, axes = plt.subplots(1, 2, figsize=(14.8, 5.4), constrained_layout=True)
ax0, ax1 = axes
full_colors = [f'C{i % 10}' for i in range(len(x_vals))]
dw_colors = [f'C{i % 10}' for i in range(len(dw_x_vals))]

for x0, color in zip(x_vals, full_colors):
    C_vec = np.clip(corr_by_x[int(x0)], 1e-300, None)
    ax0.plot(
        log_ry_full,
        np.log(C_vec),
        marker='o',
        ms=3,
        lw=1.2,
        color=color,
        label=f'x={x0}',
    )

for x0, color in zip(dw_x_vals, dw_colors):
    C_vec = np.clip(dw_corr_by_x[int(x0)], 1e-300, None)
    fit = fit_results[int(x0)]
    ax1.plot(
        log_chord_vals,
        np.log(C_vec),
        marker='o',
        ms=3,
        lw=1.0,
        color=color,
        label=rf'x={int(x0)} data',
    )
    ax1.plot(
        log_chord_vals,
        fit['fit_line'],
        lw=1.8,
        color=color,
        ls='--',
        label=rf'x={int(x0)} fit: $m={fit["slope"]:.3f}$, $R^2={fit["r2"]:.4f}$',
    )

textbox_left = (
    rf'$N_x={NX},\; N_y={NY}$' + '\n'
    + rf'$\alpha_{{\mathrm{{triv}}}}={ALPHA_1},\; \alpha_{{\mathrm{{top}}}}={ALPHA_2}$')
ax0.text(
    0.03,
    0.97,
    textbox_left,
    transform=ax0.transAxes,
    ha='left',
    va='top',
    fontsize=9,
    linespacing=1.35,
    bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='black', alpha=0.85),
)

textbox_right = (rf'fit window: $r_y \in [{FIT_RY_MIN}, {FIT_RY_MAX}]$' + '\n' 
                 + rf'DWs at $x={", ".join(map(str, (dw_loc[0], dw_loc[1])))}$')
ax1.text(
    0.03,
    0.97,
    textbox_right,
    transform=ax1.transAxes,
    ha='left',
    va='top',
    fontsize=9,
    linespacing=1.35,
    bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='black', alpha=0.85),
)

ax0.set_title(r'Fixed-$x$ squared correlations: $\log C_G$ vs $\log r_y$')
ax0.set_xlabel(r'$\log r_y$')
ax0.set_ylabel(r'$\log C_G(x_0, r_y)$')
ax0.grid(alpha=0.3)
ax0.legend(title='fixed x', fontsize=8, ncol=2, loc='lower left')

ax1.set_title(r'Domain-wall slices: $\log C_G$ vs $\log$(chord length)')
ax1.set_xlabel(r'$\log\!\left[\frac{N_y}{\pi}\sin\!\left(\pi r_y/N_y\right)\right]$')
ax1.set_ylabel(r'$\log C_G(x_{\mathrm{DW}}, r_y)$')
ax1.grid(alpha=0.3)
ax1.legend(fontsize=8, ncol=2, loc='lower left')

plt.show()

print('Domain-wall chord-length fit summary')
print('====================================')
for x0 in dw_x_vals:
    fit = fit_results[int(x0)]
    print(
        f'x={int(x0):2d}: slope={fit["slope"]:+.8f}, '
        f'intercept={fit["intercept"]:+.8f}, R^2={fit["r2"]:.8f}'
    )


In [ ]:
# Fixed-x fit diagnostics across x for the squared correlator
x_all = np.arange(NX, dtype=int)
log_chord_fit = np.log((NY / np.pi) * np.sin(np.pi * ry_vals_fit / NY))
fit_mask_all_x = (ry_vals_fit >= FIT_RY_MIN) & (ry_vals_fit <= FIT_RY_MAX)
if np.count_nonzero(fit_mask_all_x) < 2:
    raise ValueError('Need at least two points in the fit window for the fixed-x fit diagnostics.')

slope_by_x = np.full(NX, np.nan, dtype=float)
intercept_by_x = np.full(NX, np.nan, dtype=float)
r2_by_x = np.full(NX, np.nan, dtype=float)

for x0 in x_all:
    C_vec = np.clip(squared_corr_profile(Gker, x0, ry_vals_fit).real, 1e-300, None)
    x_fit = log_chord_fit[fit_mask_all_x]
    y_fit = np.log(C_vec[fit_mask_all_x])
    slope, intercept = np.polyfit(x_fit, y_fit, 1)
    y_hat = slope * x_fit + intercept
    ss_res = float(np.sum((y_fit - y_hat) ** 2))
    ss_tot = float(np.sum((y_fit - np.mean(y_fit)) ** 2))
    r2 = np.nan if ss_tot == 0.0 else 1.0 - ss_res / ss_tot
    slope_by_x[int(x0)] = float(slope)
    intercept_by_x[int(x0)] = float(intercept)
    r2_by_x[int(x0)] = float(r2)

fig, axes = plt.subplots(1, 2, figsize=(14.2, 4.6), constrained_layout=True)
ax_slope, ax_r2 = axes

dw_loc_plot = np.asarray(getattr(model, 'DW_loc', []), dtype=int) % NX
trivial_side_neighbors = np.sort(np.array([
    (dw_loc_plot[0] - 1) % NX,
    dw_loc_plot[0] % NX,
    dw_loc_plot[1] % NX,
    (dw_loc_plot[1] + 1) % NX,
], dtype=int))

ax_slope.plot(x_all, slope_by_x, marker='o', ms=4, lw=1.6, color='tab:blue')
ax_slope.axhline(-2.0, color='black', ls='--', lw=1.2, alpha=0.75)
for xdw in dw_loc_plot:
    ax_slope.axvline(int(xdw), color='tab:red', ls='--', lw=1.2, alpha=0.8)
for xedge in trivial_side_neighbors:
    ax_slope.axvspan(xedge - 0.18, xedge + 0.18, color='tab:orange', alpha=0.18)
ax_slope.set_title(rf'Fit slope vs $x$  for  $r_y \in [{FIT_RY_MIN}, {FIT_RY_MAX}]$')
ax_slope.set_xlabel(r'$x$')
ax_slope.set_ylabel(r'fitted slope of $\log C_G$ vs $\log(\mathrm{{chord}})$')
ax_slope.set_xticks(np.arange(0, NX, max(1, NX // 10)))
ax_slope.grid(alpha=0.3)

ax_r2.plot(x_all, r2_by_x, marker='o', ms=4, lw=1.6, color='tab:green')
for xdw in dw_loc_plot:
    ax_r2.axvline(int(xdw), color='tab:red', ls='--', lw=1.2, alpha=0.8)
for xedge in trivial_side_neighbors:
    ax_r2.axvspan(xedge - 0.18, xedge + 0.18, color='tab:orange', alpha=0.18)
ax_r2.set_title(rf'$R^2$ vs $x$  for  $r_y \in [{FIT_RY_MIN}, {FIT_RY_MAX}]$')
ax_r2.set_xlabel(r'$x$')
ax_r2.set_ylabel(r'$R^2$ of the linear log-chord fit')
ax_r2.set_xticks(np.arange(0, NX, max(1, NX // 10)))
ax_r2.set_ylim(min(0.0, np.nanmin(r2_by_x) - 0.02), 1.01)
ax_r2.grid(alpha=0.3)

textbox = (
    rf'fit window: $r_y \in [{FIT_RY_MIN}, {FIT_RY_MAX}]$' + '\n'
    + rf'DWs at $x={list(map(int, dw_loc_plot))}$' + '\n'
    + r'orange bands: wall + trivial-side neighbor')
ax_r2.text(
    0.03,
    0.05,
    textbox,
    transform=ax_r2.transAxes,
    ha='left',
    va='bottom',
    fontsize=9,
    linespacing=1.35,
    bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='black', alpha=0.85),
)

plt.suptitle('Fixed-$x$ log-chord fit diagnostics for the squared correlator')
plt.show()

print('Fixed-x fit summary over the log-chord window:')
print(f"{'x':>3}  {'slope':>12}  {'intercept':>12}  {'R^2':>10}")
print('-' * 43)
for x0 in x_all:
    print(f"{int(x0):3d}  {slope_by_x[int(x0)]:+12.6f}  {intercept_by_x[int(x0)]:12.6f}  {r2_by_x[int(x0)]:10.6f}")


In [ ]:
# X-averaged squared correlator and log-chord fit
x_all_avg = np.arange(NX, dtype=int)
corr_all_x = np.stack([
    squared_corr_profile(Gker, int(x0), ry_vals_full).real
    for x0 in x_all_avg
], axis=0)

C_xavg_full = np.mean(corr_all_x, axis=0)
C_xavg_fit = np.clip(C_xavg_full, 1e-300, None)

log_ry_xavg = np.log(ry_vals_full.astype(float))
log_chord_xavg = np.log((NY / np.pi) * np.sin(np.pi * ry_vals_full / NY))
fit_mask_xavg = (ry_vals_full >= FIT_RY_MIN) & (ry_vals_full <= FIT_RY_MAX)
if np.count_nonzero(fit_mask_xavg) < 2:
    raise ValueError('Need at least two points in the fit window for the x-averaged squared correlator fit.')

x_fit = log_chord_xavg[fit_mask_xavg]
y_fit = np.log(C_xavg_fit[fit_mask_xavg])
slope_xavg, intercept_xavg = np.polyfit(x_fit, y_fit, 1)
y_hat = slope_xavg * x_fit + intercept_xavg
ss_res = float(np.sum((y_fit - y_hat) ** 2))
ss_tot = float(np.sum((y_fit - np.mean(y_fit)) ** 2))
r2_xavg = np.nan if ss_tot == 0.0 else 1.0 - ss_res / ss_tot
fit_line_xavg = slope_xavg * log_chord_xavg + intercept_xavg

fig, axes = plt.subplots(1, 2, figsize=(13.8, 4.8), constrained_layout=True)
ax0, ax1 = axes

ax0.plot(
    log_ry_xavg,
    np.log(C_xavg_fit),
    marker='o',
    ms=3.5,
    lw=1.4,
    color='tab:blue',
)
ax0.set_title(r'$x$-averaged squared correlator: $\log C_{\mathrm{avg}}$ vs $\log r_y$')
ax0.set_xlabel(r'$\log r_y$')
ax0.set_ylabel(r'$\log C_{\mathrm{avg}}(r_y)$')
ax0.grid(alpha=0.3)

ax1.plot(
    log_chord_xavg,
    np.log(C_xavg_fit),
    marker='o',
    ms=3.5,
    lw=1.2,
    color='tab:blue',
    label=r'$x$-averaged data',
)
ax1.plot(
    log_chord_xavg,
    fit_line_xavg,
    ls='--',
    lw=1.8,
    color='tab:red',
    label=rf'fit: $m={slope_xavg:.4f}$, $R^2={r2_xavg:.4f}$',
)
ax1.set_title(r'$x$-averaged squared correlator: $\log C_{\mathrm{avg}}$ vs $\log(\mathrm{chord})$')
ax1.set_xlabel(r'$\log\!\left[\frac{N_y}{\pi}\sin\!\left(\pi r_y/N_y\right)\right]$')
ax1.set_ylabel(r'$\log C_{\mathrm{avg}}(r_y)$')
ax1.grid(alpha=0.3)
ax1.legend(fontsize=9)

textbox = (
    rf'$C_{{\mathrm{{avg}}}}(r_y) = \frac{{1}}{{N_x}}\sum_x C_G(x,r_y)$' + '\n'
    + rf'fit window: $r_y \in [{FIT_RY_MIN}, {FIT_RY_MAX}]$' + '\n'
    + rf'$N_x={NX},\; N_y={NY}$')
ax1.text(
    0.03,
    0.05,
    textbox,
    transform=ax1.transAxes,
    ha='left',
    va='bottom',
    fontsize=9,
    linespacing=1.35,
    bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='black', alpha=0.85),
)

plt.suptitle(r'$x$-averaged fixed-slice squared correlator')
plt.show()

print(r'$x$-averaged squared correlator fit summary')
print(f'fit window: r_y in [{FIT_RY_MIN}, {FIT_RY_MAX}]')
print(f'slope     = {slope_xavg:+.6f}')
print(f'intercept = {intercept_xavg:.6f}')
print(f'R^2       = {r2_xavg:.6f}')


## 3. Integrated entanglement contour over a fixed `x` range as a function of `A_y`

This cell keeps a user-specified contiguous `x` block and grows the `y` extent from `A_y = 1` up to `A_y = Ny`. Set `AVERAGE_OVER_ALL_Y_STARTS=True` if you want an explicit average over all possible `y` anchors; for the equilibrium DW Hamiltonian this is usually redundant because the state is uniform along `y`.

In [ ]:
# Integrated contour vs Ay for multiple x-interval sums,
# where the subsystem A is always the full-width strip
# A = {all x} x {y = 0, ..., Ay-1}

x0 = int(model.DW_loc[0]) % model.Nx
x1 = int(model.DW_loc[1]) % model.Nx

# Each tuple is (x_start, x_stop_inclusive)
X_INTERVALS = [
    (0, model.Nx - 1),
    (x0 - 1, x0 + 1),
    (x1 - 1, x1 + 1),
]

Ay_values = np.arange(2, model.Ny-1, dtype=int)
integrated_contour_by_interval = {}

x_full = np.arange(model.Nx, dtype=int)

for x_start, x_stop_inclusive in X_INTERVALS:
    x_interval = contiguous_x_block(model.Nx, x_start, x_stop_inclusive)
    curve = np.zeros_like(Ay_values, dtype=float)

    for i, Ay in enumerate(Ay_values):
        # Subsystem A is always full width in x, height Ay in y
        y_region = np.arange(Ay, dtype=int)

        _, s_map = contour_for_subsystem(model, G_eq, x_full, y_region)

        # Only after computing s_A on the full subsystem do we sum over chosen x interval
        curve[i] = float(np.sum(s_map[x_interval, :]))

    integrated_contour_by_interval[(x_start, x_stop_inclusive)] = {
        "x_interval": x_interval,
        "curve": curve,
    }


def fit_line_with_error(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if x.size < 3:
        return {
            "m": np.nan,
            "m_err": np.nan,
            "b": np.nan,
            "r2": np.nan,
            "x": x,
            "y": y,
            "y_fit": np.full_like(y, np.nan),
        }

    try:
        coeffs, cov = np.polyfit(x, y, 1, cov=True)
        m, b = float(coeffs[0]), float(coeffs[1])
        m_err = float(np.sqrt(cov[0, 0])) if np.isfinite(cov[0, 0]) else np.nan
    except Exception:
        m, b, m_err = np.nan, np.nan, np.nan

    if not np.isfinite(m):
        return {
            "m": np.nan,
            "m_err": np.nan,
            "b": np.nan,
            "r2": np.nan,
            "x": x,
            "y": y,
            "y_fit": np.full_like(y, np.nan),
        }

    y_fit = m * x + b
    ss_res = float(np.sum((y - y_fit) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = np.nan if ss_tot == 0 else (1.0 - ss_res / ss_tot)

    return {
        "m": m,
        "m_err": m_err,
        "b": b,
        "r2": r2,
        "x": x,
        "y": y,
        "y_fit": y_fit,
    }


fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), constrained_layout=True)

colors = plt.cm.viridis(np.linspace(0.08, 0.92, len(X_INTERVALS)))

# ---------------------------------
# Left: raw curves vs Ay
# ---------------------------------
for color, interval in zip(colors, X_INTERVALS):
    x_interval = integrated_contour_by_interval[interval]["x_interval"]
    curve = integrated_contour_by_interval[interval]["curve"]
    label = rf"$x \in [{int(x_interval[0])}, {int(x_interval[-1])}]$"
    axes[0].plot(Ay_values, curve, marker="o", ms=3, lw=1.5, color=color, label=label)

dw_loc = getattr(model, "DW_loc", None)
textbox = (
    rf"$N_x={model.Nx},\ N_y={model.Ny}$" "\n"
    rf"$\alpha_{{\mathrm{{triv}}}}={model.alpha_1},\ \alpha_{{\mathrm{{top}}}}={model.alpha_2}$" "\n"
    rf"DWs at $x={dw_loc}$"
)

axes[0].text(
    0.03,
    0.55,
    textbox,
    transform=axes[0].transAxes,
    ha="left",
    va="top",
    fontsize=9,
    linespacing=1.4,
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.85),
)

axes[0].set_xlabel(r"$A_y$")
axes[0].set_ylabel(r"$\sum_{x \in I_x}\sum_{y \in A} s_A(x,y)$")
axes[0].set_title("Integrated contour vs subsystem height")
axes[0].grid(alpha=0.3)
axes[0].legend(title="x interval $l$", fontsize=8)

# ---------------------------------
# Right: split branches vs log(chord) + linear fits
# ---------------------------------
mid = model.Ny / 2
mask_lower = Ay_values < mid
mask_upper = Ay_values > mid

Ay_lower = Ay_values[mask_lower]
Ay_upper = Ay_values[mask_upper]

x_lower = np.log(np.sin(np.pi * Ay_lower / model.Ny))
x_upper = np.log(np.sin(np.pi * Ay_upper / model.Ny))

fit_results = {}

for color, interval in zip(colors, X_INTERVALS):
    x_interval = integrated_contour_by_interval[interval]["x_interval"]
    curve = integrated_contour_by_interval[interval]["curve"]
    label = rf"$x \in [{int(x_interval[0])}, {int(x_interval[-1])}]$"

    lower_fit = fit_line_with_error(x_lower, curve[mask_lower])
    upper_fit = fit_line_with_error(x_upper, curve[mask_upper])

    fit_results[interval] = {
        "lower": lower_fit,
        "upper": upper_fit,
        "x_interval": x_interval,
    }

    axes[1].plot(
        lower_fit["x"],
        lower_fit["y"],
        "o",
        ms=3,
        color=color,
        label=(
            label
            + rf" lower: $m={lower_fit['m']:.4f}\pm{lower_fit['m_err']:.2e}$"
            + rf", $R^2={lower_fit['r2']:.4f}$"
        ),
    )
    axes[1].plot(
        lower_fit["x"],
        lower_fit["y_fit"],
        "-",
        lw=1.4,
        color=color,
    )

    axes[1].plot(
        upper_fit["x"],
        upper_fit["y"],
        "s",
        ms=3,
        color=color,
        label=(
            label
            + rf" upper: $m={upper_fit['m']:.4f}\pm{upper_fit['m_err']:.2e}$"
            + rf", $R^2={upper_fit['r2']:.4f}$"
        ),
    )
    axes[1].plot(
        upper_fit["x"],
        upper_fit["y_fit"],
        "--",
        lw=1.4,
        color=color,
    )

axes[1].set_xlabel(r"$\log\!\left[\sin\!\left(\pi A_y/N_y\right)\right]$")
axes[1].set_ylabel(r"$\sum_{x \in I_x}\sum_{y \in A} s_A(x,y)$")
axes[1].set_title("Same data split into lower / upper $A_y$ branches")
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8, ncol=1)

plt.show()

print("Fit summary")
print("===========")
for interval in X_INTERVALS:
    res = fit_results[interval]
    x_interval = res["x_interval"]
    lower = res["lower"]
    upper = res["upper"]

    print(f"x interval {interval} -> x_interval = {list(x_interval)}")
    print(
        "  lower branch: "
        f"slope={lower['m']:.6g}, "
        f"slope_err={lower['m_err']:.6g}, "
        f"intercept={lower['b']:.6g}, "
        f"R^2={lower['r2']:.6g}"
    )
    print(
        "  upper branch: "
        f"slope={upper['m']:.6g}, "
        f"slope_err={upper['m_err']:.6g}, "
        f"intercept={upper['b']:.6g}, "
        f"R^2={upper['r2']:.6g}"
    )

print(f"A_y scan = {Ay_values[0]} ... {Ay_values[-1]}")

In [ ]:
# Mutual-information-style contour characterization with:
# - subsystem defined on ALL x
# - y-regions A, B, C, AB, AC, BC, ABC defined as in analysis.ipynb
# - x_interval used ONLY in the post-EC sum
# - optional averaging over y0 translations

import numpy as np
import matplotlib.pyplot as plt

# ----------------------
# User controls
# ----------------------
L_A = 3
L_C = 3
AVERAGE_OVER_Y0 = False   # set False to keep a single anchor y0
Y0_FIXED = 0             # used only when AVERAGE_OVER_Y0 = False

x0 = int(model.DW_loc[0]) % model.Nx
x1 = int(model.DW_loc[1]) % model.Nx

# Each tuple is (x_start, x_stop_inclusive)
X_INTERVALS_MI = [
    (x0 - 1, x0 + 1),
    (x1 - 1, x1 + 1),
    (0, model.Nx - 1),
]

# Choose l_B values that keep all regions nonempty
LB_LIST = np.arange(L_A + L_C, model.Ny - 2 * (L_A + L_C) + 1, dtype=int)

# ----------------------
# Helpers
# ----------------------
def _wrap_block(y0, length, ny):
    y0 = int(y0) % ny
    return (np.arange(length, dtype=int) + y0) % ny

def _full_x_subsystem_entropy_partial_xsum(model, G_full, y_vals, x_interval):
    """
    Define the subsystem using:
      full x = [0, ..., Nx-1]
      y = y_vals

    Compute the EC on that full-width subsystem, then sum only over x_interval.
    """
    x_full = np.arange(model.Nx, dtype=int)
    idx = subsystem_indices(model.Nx, model.Ny, x_full, y_vals)
    G_sub = G_full[np.ix_(idx, idx)]
    s_map = model.entanglement_contour(G_sub, len(x_full), len(y_vals))  # shape (Nx, len(y_vals))
    return float(np.sum(s_map[np.asarray(x_interval, dtype=int), :]))

def _chord_length(l, L):
    return (L / np.pi) * np.sin(np.pi * l / L)

def _cross_ratio_chord(lA, lB, lC, L):
    lAB = lA + lB
    lBC = lB + lC
    return (_chord_length(lA, L) * _chord_length(lC, L)) / (_chord_length(lAB, L) * _chord_length(lBC, L))

def _fit_line_with_error(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if x.size < 3:
        return {"m": np.nan, "m_err": np.nan, "b": np.nan, "r2": np.nan, "x": x, "y": y}

    coeffs, cov = np.polyfit(x, y, 1, cov=True)
    m, b = float(coeffs[0]), float(coeffs[1])
    m_err = float(np.sqrt(cov[0, 0])) if np.isfinite(cov[0, 0]) else np.nan

    y_hat = m * x + b
    ss_res = float(np.sum((y - y_hat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = np.nan if ss_tot == 0 else (1.0 - ss_res / ss_tot)

    return {"m": m, "m_err": m_err, "b": b, "r2": r2, "x": x, "y": y}

# ----------------------
# Geometry setup
# ----------------------
NX, NY = model.Nx, model.Ny
X_LIST = np.array([_cross_ratio_chord(L_A, int(lB), L_C, NY) for lB in LB_LIST], dtype=float)
Z_LIST = np.log(1.0 / (1.0 - X_LIST))

if AVERAGE_OVER_Y0:
    Y0_LIST = np.arange(NY, dtype=int)
else:
    Y0_LIST = np.array([int(Y0_FIXED) % NY], dtype=int)

MI_RESULTS = {}

for interval in X_INTERVALS_MI:
    x_interval = contiguous_x_block(NX, interval[0], interval[1])

    i2_avg = np.zeros(len(LB_LIST), dtype=float)
    i3_avg = np.zeros(len(LB_LIST), dtype=float)
    d_avg = np.zeros(len(LB_LIST), dtype=float)

    for i_lb, lB in enumerate(LB_LIST):
        i2_y0 = np.zeros(len(Y0_LIST), dtype=float)
        i3_y0 = np.zeros(len(Y0_LIST), dtype=float)

        for j, y0_start in enumerate(Y0_LIST):
            yA = _wrap_block(y0_start, L_A, NY)
            yB = _wrap_block(y0_start + L_A, int(lB), NY)
            yC = _wrap_block(y0_start + L_A + int(lB), L_C, NY)

            S_A = _full_x_subsystem_entropy_partial_xsum(model, G_eq, yA, x_interval)
            S_B = _full_x_subsystem_entropy_partial_xsum(model, G_eq, yB, x_interval)
            S_C = _full_x_subsystem_entropy_partial_xsum(model, G_eq, yC, x_interval)

            S_AB = _full_x_subsystem_entropy_partial_xsum(model, G_eq, np.concatenate([yA, yB]), x_interval)
            S_AC = _full_x_subsystem_entropy_partial_xsum(model, G_eq, np.concatenate([yA, yC]), x_interval)
            S_BC = _full_x_subsystem_entropy_partial_xsum(model, G_eq, np.concatenate([yB, yC]), x_interval)
            S_ABC = _full_x_subsystem_entropy_partial_xsum(model, G_eq, np.concatenate([yA, yB, yC]), x_interval)

            i2_y0[j] = S_A + S_C - S_AC
            i3_y0[j] = S_A + S_B + S_C - S_AB - S_AC - S_BC + S_ABC

        i2_avg[i_lb] = float(np.mean(i2_y0))
        i3_avg[i_lb] = float(np.mean(i3_y0))
        d_avg[i_lb] = i2_avg[i_lb] - i3_avg[i_lb]

    fit = _fit_line_with_error(Z_LIST, d_avg)

    MI_RESULTS[interval] = {
        "x_interval": x_interval,
        "x": X_LIST.copy(),
        "z": Z_LIST.copy(),
        "lB": LB_LIST.copy(),
        "i2_avg": i2_avg,
        "i3_avg": i3_avg,
        "d_avg": d_avg,
        "fit": fit,
    }

# ----------------------
# Labels and colors
# ----------------------
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(X_INTERVALS_MI)))

def _interval_label(interval):
    xs = MI_RESULTS[interval]["x_interval"]
    return rf"$x \in [{int(xs[0])}, {int(xs[-1])}]$"

mode_label = "avg over all $y_0$" if AVERAGE_OVER_Y0 else rf"fixed $y_0={int(Y0_LIST[0])}$"

# ----------------------
# Plot 1: all x-intervals together
# ----------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

for color, interval in zip(colors, X_INTERVALS_MI):
    res = MI_RESULTS[interval]
    label = _interval_label(interval)

    axes[0].plot(res["x"], res["i2_avg"], "o-", lw=1.2, ms=4, color=color, alpha=0.9, label=label + r" : $I_2$")
    axes[0].plot(res["x"], res["i3_avg"], "s--", lw=1.2, ms=4, color=color, alpha=0.9, label=label + r" : $I_3$")
    axes[0].plot(res["x"], res["d_avg"], "^-", lw=1.4, ms=4, color=color, alpha=0.9, label=label + r" : $I_2-I_3$")

    fit = res["fit"]
    axes[1].plot(res["z"], res["d_avg"], "o", ms=4, color=color, alpha=0.9)
    if np.isfinite(fit["m"]) and np.isfinite(fit["b"]) and fit["x"].size > 1:
        xx = np.linspace(np.min(fit["x"]), np.max(fit["x"]), 200)
        yy = fit["m"] * xx + fit["b"]
        axes[1].plot(
            xx,
            yy,
            "-",
            lw=1.5,
            color=color,
            label=label + rf": $m={fit['m']:.4g}\pm{fit['m_err']:.2g}$, $R^2={fit['r2']:.4f}$",
        )

axes[0].set_xlabel("cross ratio $x$")
axes[0].set_ylabel("contour-based information")
axes[0].set_title(r"$I_2$, $I_3$, and $I_2-I_3$ vs cross ratio")
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize=7, ncol=1)

axes[1].set_xlabel(r"$\log(1/(1-x))$")
axes[1].set_ylabel(r"$I_2-I_3$")
axes[1].set_title(r"Linear fit of $I_2-I_3$ vs $\log(1/(1-x))$")
axes[1].grid(alpha=0.25)
axes[1].legend(fontsize=7, ncol=1)

textbox = (
    rf"$N_x={NX},\ N_y={NY}$" "\n"
    rf"$L_A={L_A},\ L_C={L_C}$" "\n"
    rf"{mode_label}"
)
axes[0].text(
    0.03,
    0.97,
    textbox,
    transform=axes[0].transAxes,
    ha="left",
    va="top",
    fontsize=9,
    linespacing=1.4,
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.85),
)

plt.show()

# ----------------------
# Plot 2: all x-intervals together vs l_B
# ----------------------
fig, ax = plt.subplots(1, 1, figsize=(10, 5), constrained_layout=True)

for color, interval in zip(colors, X_INTERVALS_MI):
    res = MI_RESULTS[interval]
    label = _interval_label(interval)
    ax.plot(res["lB"], res["i2_avg"], "o-", lw=1.2, ms=4, color=color, alpha=0.65, label=label + r" : $I_2$")
    ax.plot(res["lB"], res["i3_avg"], "s--", lw=1.2, ms=4, color=color, alpha=0.65, label=label + r" : $I_3$")
    ax.plot(res["lB"], res["d_avg"], "^-", lw=1.5, ms=4, color=color, alpha=1.0, label=label + r" : $I_2-I_3$")

ax.set_xlabel(r"$l_B$")
ax.set_ylabel("contour-based information")
ax.set_title(r"$I_2$, $I_3$, and $I_2-I_3$ vs separation $l_B$")
ax.grid(alpha=0.25)
ax.legend(fontsize=7, ncol=2)

plt.show()

# ----------------------
# Summary
# ----------------------
lines = []
lines.append("Contour-based MI summary")
lines.append("========================")
lines.append("Subsystems are full-width in x; x_interval only restricts the final EC sum.")
lines.append(f"AVERAGE_OVER_Y0={AVERAGE_OVER_Y0}, Y0_FIXED={Y0_FIXED if not AVERAGE_OVER_Y0 else 'all'}")
lines.append(f"L_A={L_A}, L_C={L_C}, LB_LIST={LB_LIST.tolist()}")
lines.append("")
for interval in X_INTERVALS_MI:
    res = MI_RESULTS[interval]
    fit = res["fit"]
    lines.append(f"x interval {interval} -> x-sum interval = {res['x_interval'].tolist()}")
    lines.append(
        f"  slope={fit['m']:.8e}, slope_err={fit['m_err']:.8e}, "
        f"intercept={fit['b']:.8e}, R^2={fit['r2']:.8f}"
    )
    lines.append("")
print("\n".join(lines))


## 4. Full-width strip entropy: direct vs contour-summed, and `A_y` cutoff sensitivity

This cell checks two things on the exact equilibrium DW GS:

1. Whether summing the entanglement contour over the full-width strip reproduces the direct subsystem entropy.
2. Whether excluding small-`A_y` points changes the lower-branch log-chord fit enough to explain the collaborator mismatch.


In [ ]:
def direct_strip_entropy(model, G_full, y_vals):
    """Direct von Neumann entropy of the full-width strip A = all x times y_vals."""
    x_full = np.arange(model.Nx, dtype=int)
    idx = subsystem_indices(model.Nx, model.Ny, x_full, y_vals)
    G_sub = reduced_covariance(G_full, idx)
    occ = 0.5 * (np.eye(G_sub.shape[0], dtype=np.complex128) + G_sub)
    occ = 0.5 * (occ + occ.conj().T)
    evals = np.linalg.eigvalsh(occ)
    evals = np.clip(np.real_if_close(evals), 1e-12, 1 - 1e-12)
    return float(np.sum(-(evals * np.log(evals) + (1.0 - evals) * np.log(1.0 - evals))))


def contour_summed_strip_entropy(model, G_full, y_vals):
    """Same strip entropy reconstructed by summing the contour over all x in A."""
    x_full = np.arange(model.Nx, dtype=int)
    _, s_map = contour_for_subsystem(model, G_full, x_full, y_vals)
    return float(np.sum(s_map))


def lower_branch_fit_for_cutoff(curve, Ay_values, ny, Ay_min):
    mask = (Ay_values >= int(Ay_min)) & (Ay_values < (ny / 2.0))
    x = np.log(np.sin(np.pi * Ay_values[mask] / ny))
    return fit_line_with_error(x, curve[mask])


AY_ALL = np.arange(1, model.Ny, dtype=int)
S_direct = np.zeros_like(AY_ALL, dtype=float)
S_contour = np.zeros_like(AY_ALL, dtype=float)

for i, Ay in enumerate(AY_ALL):
    y_region = np.arange(Ay, dtype=int)
    S_direct[i] = direct_strip_entropy(model, G_eq, y_region)
    S_contour[i] = contour_summed_strip_entropy(model, G_eq, y_region)

entropy_delta = S_contour - S_direct
print(f"max |S_contour - S_direct| over all Ay = {np.max(np.abs(entropy_delta)):.3e}")
print(f"rms |S_contour - S_direct| over all Ay = {np.sqrt(np.mean(entropy_delta**2)):.3e}")

FIT_WINDOWS = [
    {"title": "My script lower branch", "Ay_min": 3},
    {"title": "Collaborator lower branch", "Ay_min": 1},
]

print("\nLower-branch fits of full-width strip entropy vs log(sin(pi Ay / Ny))")
for cfg in FIT_WINDOWS:
    fit_direct = lower_branch_fit_for_cutoff(S_direct, AY_ALL, model.Ny, cfg["Ay_min"])
    fit_contour = lower_branch_fit_for_cutoff(S_contour, AY_ALL, model.Ny, cfg["Ay_min"])
    print(
        f"{cfg['title']} (Ay >= {cfg['Ay_min']}): direct slope={fit_direct['m']:.8f} +/- {fit_direct['m_err']:.3e}, "
        f"contour slope={fit_contour['m']:.8f} +/- {fit_contour['m_err']:.3e}, "
        f"|dm|={abs(fit_direct['m'] - fit_contour['m']):.3e}, "
        f"R2_direct={fit_direct['r2']:.8f}, R2_contour={fit_contour['r2']:.8f}"
    )

fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.2), constrained_layout=True)
log_sin_all = np.log(np.sin(np.pi * AY_ALL / model.Ny))

for ax, cfg in zip(axes, FIT_WINDOWS):
    mask = (AY_ALL >= int(cfg["Ay_min"])) & (AY_ALL < (model.Ny / 2.0))
    x_fit = log_sin_all[mask]
    y_direct = S_direct[mask]
    y_contour = S_contour[mask]

    fit_direct = fit_line_with_error(x_fit, y_direct)
    fit_contour = fit_line_with_error(x_fit, y_contour)

    ax.scatter(x_fit, y_direct, s=28, color="black", alpha=0.85, label="direct full-width sum")
    ax.scatter(x_fit, y_contour, s=28, facecolors="none", edgecolors="tab:red", linewidths=1.2, label="contour full-width sum")
    ax.plot(
        fit_direct["x"],
        fit_direct["y_fit"],
        "-",
        lw=1.8,
        color="black",
        label=rf"direct fit: $m={fit_direct['m']:.4f} \pm {fit_direct['m_err']:.2e}$, $R^2={fit_direct['r2']:.6f}$",
    )
    ax.plot(
        fit_contour["x"],
        fit_contour["y_fit"],
        "--",
        lw=1.8,
        color="tab:red",
        label=rf"contour fit: $m={fit_contour['m']:.4f} \pm {fit_contour['m_err']:.2e}$, $R^2={fit_contour['r2']:.6f}$",
    )

    ax.set_title(rf"{cfg['title']} ($A_y \geq {cfg['Ay_min']}$, full width)")
    ax.set_xlabel(r"$\log[\sin(\pi A_y / N_y)]$")
    ax.set_ylabel(r"$S_A$ for full-width strip")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.show()
